
# Extract learned physical parameters from a trained model

This example teaches you how to use GeoPrior's
``extract_physical_parameters`` helper.

Unlike the publication-oriented scripts in ``figure_generation/``,
this function is a compact model-inspection and export utility.
It extracts the learned physical parameters from a trained PINN-style
model and can optionally write them to CSV.

## Why this matters
After training, you often want a compact answer to questions like:

- what value did the model learn for ``mv``?
- what is the effective ``kappa``?
- what fixed constants are active?
- what are the mean learned field values for ``K``, ``Ss``, and
  ``tau``?

This helper turns those checks into one small inspection step.


## Imports
We call the real helper from the package.
For the lesson page, we use a lightweight mock GeoPrior-like model
that exposes the same attributes and methods the extractor expects.



In [ ]:
from __future__ import annotations

import tempfile
from pathlib import Path

import pandas as pd
import tensorflow as tf

from geoprior.models import extract_physical_parameters

## Build a tiny GeoPrior-like inspection model
The real helper looks for:

- current_mv() or log_mv / _mv_fixed
- current_kappa() or log_kappa / _kappa_fixed
- gamma_w
- h_ref

and, for GeoPrior-style models with ``inputs`` provided:

- model(inputs, training=False)
- model.split_physics_predictions(outputs)

So the lesson uses a tiny mock model that exposes exactly those
pieces and returns small synthetic fields.



In [ ]:
class DemoGeoPriorModel:
    def __init__(self):
        self._mv = tf.constant(2.5e-7, dtype=tf.float32)
        self._kappa = tf.constant(1.15, dtype=tf.float32)
        self.gamma_w = tf.constant(9810.0, dtype=tf.float32)
        self.h_ref = tf.constant(0.0, dtype=tf.float32)

    def current_mv(self):
        return self._mv

    def current_kappa(self):
        return self._kappa

    def __call__(self, inputs, training=False):
        coords = tf.convert_to_tensor(inputs["coords"], tf.float32)
        x = coords[..., 1:2]
        y = coords[..., 2:3]

        # Small synthetic mean-path outputs.
        s_mean = 0.01 * x + 0.02 * y
        h_mean = 0.03 * x - 0.01 * y

        # Small synthetic physics fields.
        K_field = 1e-4 * tf.exp(0.02 * x)
        Ss_field = 5e-6 * tf.exp(0.01 * y)
        tau_field = 2.0e7 + 3.0e5 * x

        return {
            "subs_mean": s_mean,
            "head_mean": h_mean,
            "K_field": K_field,
            "Ss_field": Ss_field,
            "tau_field": tau_field,
        }

    def split_physics_predictions(self, outputs):
        return (
            outputs["subs_mean"],
            outputs["head_mean"],
            outputs["K_field"],
            outputs["Ss_field"],
            outputs["tau_field"],
        )


model = DemoGeoPriorModel()

## Build compact synthetic GeoPrior inputs
For GeoPrior-style field summaries, the extractor only needs inputs
that the model's forward pass can consume.

Here we provide a tiny dict-input batch with:

- coords shaped (B, H, 3)
- and a placeholder H_field because GeoPrior-style batches often
  carry thickness as well.



In [ ]:
coords = tf.constant(
    [
        [
            [2024.0, 100.0, 35.0],
            [2025.0, 101.0, 35.2],
            [2026.0, 102.0, 35.4],
        ],
        [
            [2024.0, 104.0, 36.0],
            [2025.0, 105.0, 36.3],
            [2026.0, 106.0, 36.5],
        ],
    ],
    dtype=tf.float32,
)

inputs = {
    "coords": coords,
    "H_field": tf.ones((2, 3, 1), dtype=tf.float32) * 18.0,
    "dynamic_features": tf.zeros((2, 3, 4), dtype=tf.float32),
}

print("coords shape:", coords.shape)

## Extract the compact scalar summary
Because ``model_name='geoprior'`` and ``inputs`` are provided,
the extractor will:

- read mv and kappa through their accessor methods,
- read gamma_w and h_ref directly,
- run a forward pass,
- summarize the K, Ss, and tau fields using the configured reducer.



In [ ]:
params = extract_physical_parameters(
    model,
    model_name="geoprior",
    inputs=inputs,
    verbose=1,
)

print("")
print("Extracted parameter summary")
for k, v in params.items():
    print(f"{k}: {v}")

## Request the full fields as well
``return_fields=True`` keeps the compact scalar summary but also
returns the raw field arrays in the output dictionary.

This is useful when you want both:

- a compact report,
- and a quick way to inspect or save the actual field tensors.



In [ ]:
params_with_fields = extract_physical_parameters(
    model,
    model_name="geoprior",
    inputs=inputs,
    return_fields=True,
)

print("")
print("Field-bearing keys")
for k in params_with_fields:
    if k.endswith("_field"):
        v = params_with_fields[k]
        print(k, "shape ->", getattr(v, "shape", None))

## Export the result to CSV
The real helper can export the parameter summary directly to CSV.
We demonstrate that with an explicit filename and output directory.



In [ ]:
tmp_dir = Path(
    tempfile.mkdtemp(prefix="gp_sg_extract_params_")
)
csv_name = "geoprior_physical_parameters.csv"

_ = extract_physical_parameters(
    model,
    model_name="geoprior",
    inputs=inputs,
    to_csv=True,
    filename=csv_name,
    save_dir=str(tmp_dir),
    verbose=1,
)

csv_path = tmp_dir / csv_name
csv_tab = pd.read_csv(csv_path)

print("")
print("Written CSV")
print(" -", csv_path)

print("")
print("CSV preview")
print(csv_tab.to_string(index=False))

## Learn how to read the output
A useful reading order is:

1. inspect the scalar control parameters:

   - ``Compressibility_mv``
   - ``Consistency_Kappa``
   - ``Unit_Weight_Water_gamma_w``
   - ``Reference_Head_h_ref``

2. then inspect the summarized field parameters:

   - ``Hydraulic_Conductivity_K_mean``
   - ``Specific_Storage_Ss_mean``
   - ``Relaxation_Time_tau_mean``

That gives a compact physical snapshot of the trained model.



## Why GeoPrior uses inputs for field summaries
For GeoPrior-style models, K, Ss, and tau are not stored as single
global scalars. They are learned fields produced by the forward pass.

That is why the extractor needs ``inputs`` when you want those field
summaries:

- it runs the model,
- splits the physics predictions,
- and reduces the resulting fields.



## Why this helper is still inspection, not evaluation
This function does not compute residual maps, losses, or validation
metrics.

Its job is narrower:

- extract learned physical parameters,
- summarize learned fields,
- optionally export them.

So it is the natural final page in the ``model_inspection`` section.



## Practical takeaway
A useful workflow after training is:

1. inspect history with:

   - ``plot_history_in(...)``
   - ``plot_epsilons_in(...)``
   - ``plot_physics_losses_in(...)``

2. inspect payload values with:

   - ``plot_physics_values_in(...)``

3. finish with:

   - ``extract_physical_parameters(...)``

That gives a compact end-to-end inspection path from optimization
dynamics to learned physical meaning.
